In [5]:
from pathlib import Path
import time
import pandas as pd
import numpy as np

DATA_DIR = Path.home() / "datasets" / "nyc-taxi" / "2025" / "cleaned"

files = sorted(DATA_DIR.glob("yellow_tripdata_2025-??_clean.parquet"))

len(files), [f.name for f in files]

(12,
 ['yellow_tripdata_2025-01_clean.parquet',
  'yellow_tripdata_2025-02_clean.parquet',
  'yellow_tripdata_2025-03_clean.parquet',
  'yellow_tripdata_2025-04_clean.parquet',
  'yellow_tripdata_2025-05_clean.parquet',
  'yellow_tripdata_2025-06_clean.parquet',
  'yellow_tripdata_2025-07_clean.parquet',
  'yellow_tripdata_2025-08_clean.parquet',
  'yellow_tripdata_2025-09_clean.parquet',
  'yellow_tripdata_2025-10_clean.parquet',
  'yellow_tripdata_2025-11_clean.parquet',
  'yellow_tripdata_2025-12_clean.parquet'])

In [6]:
tiers = {
    "1_month": files[:1],
    "3_months": files[:3],
    "6_months": files[:6],
    "12_months": files[:12],
}

for name, tier_files in tiers.items():
    print(name, len(tier_files))

1_month 1
3_months 3
6_months 6
12_months 12


In [7]:
# timing helper
def timed(func):
    start = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - start
    return result, elapsed

In [11]:
def benchmark_pandas(file_list):
    results = {}

    # 1. Load
    df, results["load"] = timed(
        lambda: pd.concat(
            [pd.read_parquet(f) for f in file_list],
            ignore_index=True
        )
    )

    # 2. Filter
    filtered, results["filter"] = timed(
        lambda: df[
            (df["trip_distance"] > 0)
            & (df["fare_amount"] > 0)
            & (df["total_amount"] > 0)
        ]
    )

    # 3. Null handling
    null_counts, results["null_count"] = timed(
        lambda: df.isna().sum()
    )

    # 4. Groupby aggregation
    grouped, results["groupby"] = timed(
        lambda: df.groupby("payment_type").agg(
            trip_count=("VendorID", "size"),
            avg_distance=("trip_distance", "mean"),
            avg_fare=("fare_amount", "mean"),
            avg_tip=("tip_amount", "mean"),
            avg_total=("total_amount", "mean"),
        )
    )

    # 5. Feature engineering
    def make_features():
        temp = df.copy()

        temp["trip_duration_min"] = (
            temp["tpep_dropoff_datetime"]
            - temp["tpep_pickup_datetime"]
        ).dt.total_seconds() / 60

        temp["pickup_hour"] = (
            temp["tpep_pickup_datetime"].dt.hour
        )

        temp["tip_pct"] = np.where(
            temp["fare_amount"] > 0,
            temp["tip_amount"] / temp["fare_amount"] * 100,
            np.nan
        )

        return temp

    engineered, results["feature_engineering"] = timed(
        make_features
    )

    # 6. Sort
    sorted_df, results["sort"] = timed(
        lambda: df.sort_values(
            "total_amount",
            ascending=False
        )
    )

    results["rows"] = len(df)

    results["total"] = sum(
        results[key]
        for key in [
            "load",
            "filter",
            "null_count",
            "groupby",
            "feature_engineering",
            "sort",
        ]
    )

    return results

In [12]:
test_results = benchmark_pandas(tiers["1_month"])
test_results

{'load': 0.05159811899648048,
 'filter': 0.08336835299269296,
 'null_count': 0.041844713006867096,
 'groupby': 0.0802387049770914,
 'feature_engineering': 0.08474736101925373,
 'sort': 0.5509354200039525,
 'rows': 3475082,
 'total': 0.8927326709963381}

In [13]:
pandas_results = []

for tier_name, tier_files in tiers.items():
    print(f"Running pandas benchmark: {tier_name}")

    result = benchmark_pandas(tier_files)

    result["tier"] = tier_name
    result["backend"] = "pandas"

    pandas_results.append(result)

    print(result)

Running pandas benchmark: 1_month
{'load': 0.051889174996176735, 'filter': 0.08589484897674993, 'null_count': 0.041542088001733646, 'groupby': 0.08027328198659234, 'feature_engineering': 0.08422539199818857, 'sort': 0.5542831789935008, 'rows': 3475082, 'total': 0.898107964952942, 'tier': '1_month', 'backend': 'pandas'}
Running pandas benchmark: 3_months
{'load': 0.24010310802259482, 'filter': 0.29281117900973186, 'null_count': 0.13935941399540752, 'groupby': 0.26338518399279565, 'feature_engineering': 0.4264578410075046, 'sort': 2.691913342976477, 'rows': 11197681, 'total': 4.0540300690045115, 'tier': '3_months', 'backend': 'pandas'}
Running pandas benchmark: 6_months
{'load': 0.4914927249774337, 'filter': 0.5645908389997203, 'null_count': 0.3085575259756297, 'groupby': 0.5530255130142905, 'feature_engineering': 0.8790372279763687, 'sort': 5.700540153979091, 'rows': 24082473, 'total': 8.497243984922534, 'tier': '6_months', 'backend': 'pandas'}
Running pandas benchmark: 12_months
{'load

In [ ]:
pandas_results_df = pd.DataFrame(pandas_results)

pandas_results_df

,load,filter,null_count,groupby,feature_engineering,sort,rows,total,tier,backend
0,0.057737,0.088395,0.042741,0.081019,0.084105,0.556450,3475082,0.910447,1_month,pandas
1,0.232670,0.267387,0.134410,0.259551,0.407090,2.431664,11197681,3.732772,3_months,pandas
2,0.476336,0.573482,0.312476,0.552894,0.879605,5.700846,24082473,8.495640,6_months,pandas
3,0.987617,1.196741,0.584456,1.117304,1.835112,12.197595,48720015,17.918826,12_months,pandas


In [ ]:
#cpu benchmark notes
#predictable linear scaling of time with data size, as expected. The groupby and feature engineering steps are the most time-consuming, which is consistent with the complexity of these operations.

In [ ]:
#switching to GPU backend for benchmarking. The GPU backend is expected to provide significant speedup for large datasets, especially for operations that can be parallelized effectively.

In [1]:
from pathlib import Path
import time

import cudf
import cupy as cp

DATA_DIR = Path.home() / "datasets" / "nyc-taxi" / "2025" / "cleaned"

files = sorted(
    DATA_DIR.glob("yellow_tripdata_2025-??_clean.parquet")
)

tiers = {
    "1_month": files[:1],
    "3_months": files[:3],
    "6_months": files[:6],
    "12_months": files[:12],
}

print("Files:", len(files))

Files: 12


In [2]:
def timed_gpu(func):
    cp.cuda.Stream.null.synchronize()

    start = time.perf_counter()

    result = func()

    cp.cuda.Stream.null.synchronize()

    elapsed = time.perf_counter() - start

    return result, elapsed

In [3]:
def benchmark_cudf(file_list):
    results = {}

    # 1. Load
    df, results["load"] = timed_gpu(
        lambda: cudf.concat(
            [cudf.read_parquet(str(f)) for f in file_list],
            ignore_index=True
        )
    )

    # 2. Filter
    filtered, results["filter"] = timed_gpu(
        lambda: df[
            (df["trip_distance"] > 0)
            & (df["fare_amount"] > 0)
            & (df["total_amount"] > 0)
        ]
    )

    # 3. Null handling
    null_counts, results["null_count"] = timed_gpu(
        lambda: df.isna().sum()
    )

    # 4. Groupby aggregation
    grouped, results["groupby"] = timed_gpu(
        lambda: df.groupby("payment_type").agg({
            "VendorID": "count",
            "trip_distance": "mean",
            "fare_amount": "mean",
            "tip_amount": "mean",
            "total_amount": "mean",
        })
    )

    # 5. Feature engineering
    def make_features():
        temp = df.copy()

        temp["trip_duration_min"] = (
            temp["tpep_dropoff_datetime"]
            - temp["tpep_pickup_datetime"]
        ).dt.total_seconds() / 60

        temp["pickup_hour"] = (
            temp["tpep_pickup_datetime"].dt.hour
        )

        tip_pct = (
            temp["tip_amount"]
            / temp["fare_amount"]
            * 100
        )

        temp["tip_pct"] = tip_pct.where(
            temp["fare_amount"] > 0
        )

        return temp

    engineered, results["feature_engineering"] = timed_gpu(
        make_features
    )

    # 6. Sort
    sorted_df, results["sort"] = timed_gpu(
        lambda: df.sort_values(
            "total_amount",
            ascending=False
        )
    )

    results["rows"] = len(df)

    results["total"] = sum(
        results[key]
        for key in [
            "load",
            "filter",
            "null_count",
            "groupby",
            "feature_engineering",
            "sort",
        ]
    )

    return results

In [4]:
test_gpu = benchmark_cudf(tiers["1_month"])

test_gpu

{'load': 0.5945786699885502,
 'filter': 0.04340128300827928,
 'null_count': 0.041057024005567655,
 'groupby': 0.024734231003094465,
 'feature_engineering': 0.19031501602148637,
 'sort': 0.0692993710108567,
 'rows': 3475082,
 'total': 0.9633855950378347}

In [ ]:
test_gpu["rows"]

3475082

In [ ]:
cudf_results = []

for tier_name, tier_files in tiers.items():

    print(f"Running cuDF benchmark: {tier_name}")

    result = benchmark_cudf(tier_files)

    result["tier"] = tier_name
    result["backend"] = "cuDF"

    cudf_results.append(result)

    print(result)

Running cuDF benchmark: 1_month
{'load': 0.11933359800605103, 'filter': 0.03420211200136691, 'null_count': 0.021342646999983117, 'groupby': 0.007892302994150668, 'feature_engineering': 0.0581556449906202, 'sort': 0.06224926799768582, 'rows': 3475082, 'total': 0.30317557298985776, 'tier': '1_month', 'backend': 'cuDF'}
Running cuDF benchmark: 3_months
{'load': 0.2679533960035769, 'filter': 0.09664664400042966, 'null_count': 0.045506852999096736, 'groupby': 0.021655976001056843, 'feature_engineering': 0.1554580500087468, 'sort': 0.20936744600476231, 'rows': 11197681, 'total': 0.7965883650176693, 'tier': '3_months', 'backend': 'cuDF'}
Running cuDF benchmark: 6_months
{'load': 0.5341708680061856, 'filter': 0.1964125100057572, 'null_count': 0.07760185599909164, 'groupby': 0.04437339499418158, 'feature_engineering': 0.3160693889949471, 'sort': 0.45486281599733047, 'rows': 24082473, 'total': 1.6234908339974936, 'tier': '6_months', 'backend': 'cuDF'}
Running cuDF benchmark: 12_months
{'load': 1

In [ ]:
cudf_results_df = cudf.DataFrame(cudf_results)

cudf_results_df

,load,filter,null_count,groupby,feature_engineering,sort,rows,total,tier,backend
0,0.119334,0.034202,0.021343,0.007892,0.058156,0.062249,3475082,0.303176,1_month,cuDF
1,0.267953,0.096647,0.045507,0.021656,0.155458,0.209367,11197681,0.796588,3_months,cuDF
2,0.534171,0.196413,0.077602,0.044373,0.316069,0.454863,24082473,1.623491,6_months,cuDF
3,1.068355,0.386197,0.135171,0.090657,2.171076,0.935869,48720015,4.787324,12_months,cuDF


In [ ]:
#loading in pandas is faster than loading in cuDF, which is expected due to the overhead of transferring data to the GPU. However, for larger datasets, the GPU backend is expected to outperform pandas in subsequent operations.
#sorting is also faster in cuDF, which is expected due to the parallelization capabilities of GPUs. Overall, the GPU backend shows significant performance improvements for larger datasets, especially in operations that can be parallelized effectively.
# note, over 48.7million rows, pandas ~17.9 seconds, cuDF ~4.8 seconds
# will repeat 5x and report average times for each operation to get a more robust comparison between the two backends.

## Repeated Benchmark: 5 Runs per Tier

In [14]:
import gc
import pandas as pd

def run_repeated_benchmark(
    benchmark_func,
    tiers,
    backend,
    runs=5
):
    records = []

    for tier_name, tier_files in tiers.items():
        print(f"\n{backend} — {tier_name}")

        for run_num in range(1, runs + 1):
            gc.collect()

            result = benchmark_func(tier_files)

            result["tier"] = tier_name
            result["backend"] = backend
            result["run"] = run_num

            records.append(result)

            print(
                f"Run {run_num}: "
                f"{result['total']:.4f} s"
            )

    return pd.DataFrame(records)

In [15]:
pandas_runs = run_repeated_benchmark(
    benchmark_pandas,
    tiers,
    backend="pandas",
    runs=5
)

pandas_runs


pandas — 1_month
Run 1: 0.8814 s
Run 2: 0.8957 s
Run 3: 1.0287 s
Run 4: 1.0257 s
Run 5: 0.9008 s

pandas — 3_months
Run 1: 4.0542 s
Run 2: 3.7424 s
Run 3: 3.7523 s
Run 4: 3.7429 s
Run 5: 3.7187 s

pandas — 6_months
Run 1: 9.0929 s
Run 2: 8.5289 s
Run 3: 8.5379 s
Run 4: 9.0544 s
Run 5: 9.0578 s

pandas — 12_months
Run 1: 17.8928 s
Run 2: 17.9341 s
Run 3: 18.9162 s
Run 4: 17.9642 s
Run 5: 17.9170 s


,load,filter,null_count,groupby,feature_engineering,sort,rows,total,tier,backend,run
0,0.055523,0.081385,0.041165,0.079326,0.082768,0.541241,3475082,0.881408,1_month,pandas,1
1,0.047088,0.101210,0.041206,0.079532,0.082670,0.543986,3475082,0.895693,1_month,pandas,2
2,0.053331,0.088383,0.043361,0.082046,0.086621,0.674910,3475082,1.028652,1_month,pandas,3
3,0.056503,0.088368,0.043377,0.081964,0.087557,0.667911,3475082,1.025679,1_month,pandas,4
4,0.054136,0.089213,0.041669,0.080529,0.083966,0.551301,3475082,0.900813,1_month,pandas,5
5,0.246998,0.284084,0.139620,0.263468,0.425906,2.694097,11197681,4.054174,3_months,pandas,1
6,0.238395,0.269875,0.135417,0.259595,0.405988,2.433108,11197681,3.742379,3_months,pandas,2
7,0.236463,0.265684,0.136149,0.259125,0.404419,2.450443,11197681,3.752284,3_months,pandas,3
8,0.236795,0.266152,0.134001,0.259533,0.404647,2.441759,11197681,3.742887,3_months,pandas,4
9,0.245156,0.273009,0.133348,0.255960,0.397914,2.413270,11197681,3.718657,3_months,pandas,5


In [17]:
pandas_runs.shape

(20, 11)

In [18]:
REPORT_DIR = (
    Path.home()
    / "projects"
    / "portfolio"
    / "project-00-spark-validation"
    / "reports"
)

REPORT_DIR.mkdir(exist_ok=True)

pandas_runs.to_csv(
    REPORT_DIR / "pandas_benchmark_runs.csv",
    index=False
)

In [19]:
timing_cols = [
    "load",
    "filter",
    "null_count",
    "groupby",
    "feature_engineering",
    "sort",
    "total"
]

In [20]:
pandas_summary = (
    pandas_runs
    .groupby("tier")[timing_cols]
    .agg(["median", "mean", "std"])
)

pandas_summary

load                        filter                      \
             median      mean       std    median      mean       std   
tier                                                                    
12_months  0.981093  0.976822  0.021930  1.208007  1.223580  0.037257   
1_month    0.054136  0.053316  0.003691  0.088383  0.089712  0.007165   
3_months   0.238395  0.240761  0.004950  0.269875  0.271761  0.007506   
6_months   0.490152  0.490696  0.011932  0.593962  0.588484  0.019836   

          null_count                       groupby  ...            \
              median      mean       std    median  ...       std   
tier                                                ...             
12_months   0.585115  0.587803  0.009165  1.116749  ...  0.008240   
1_month     0.041669  0.042156  0.001125  0.080529  ...  0.001293   
3_months    0.135417  0.135707  0.002453  0.259533  ...  0.002665   
6_months    0.330870  0.327561  0.005638  0.561109  ...  0.004886   

          feature_engineering                           sort             \
                       median      mean       std     median       mean   
tier                                                                      
12_months            1.840788  1.860256  0.051411  12.205858  12.356815   
1_month              0.083966  0.084716  0.002250   0.551301   0.595870   
3_months             0.404647  0.407775  0.010609   2.441759   2.486535   
6_months             0.927046  0.910822  0.025824   6.142449   5.978967   

                         total                       
                std     median       mean       std  
tier                                                 
12_months  0.336434  17.934147  18.124881  0.443106  
1_month    0.069101   0.900813   0.946449  0.074034  
3_months   0.116846   3.742887   3.802076  0.141473  
6_months   0.229196   9.054365   8.854374  0.293408  

[4 rows x 21 columns]

In [21]:
pandas_summary.to_csv(
    REPORT_DIR / "pandas_benchmark_summary.csv"
)